# Stage 8: Model Comparison, Decision Layer and Selection

Final stage. Load the saved models, no retraining. Stage 7 left XGBoost, LightGBM and
Logistic Regression bunched together, none clearly beating RFM. When models are this
close, cost should break the tie, not the leaderboard.

Goal: a final answer. Which model, if any, should the retention team use, at what
threshold, and does it actually save enough money to be worth it.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

# Data directory
DATA_DIR = Path(PROJECT_ROOT, "data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
import joblib
import numpy as np
import pandas as pd

from src.config import (
    COST_MISSED_CHURNER,
    COST_WASTED_OFFER,
    CV_FOLDS,
    FEATURE_COLUMNS,
    MIN_COST_REDUCTION_VS_TRIVIAL,
    MODEL_SELECTION_JSON,
    MODELS_DIR,
    OUTPUT_DIR,
    RANDOM_SEED,
    TEST_PARQUET,
    TRAIN_PARQUET,
)
from src.logger import setup_logger

logger = setup_logger("06-comparison-threshold")

logger.info("COST_MISSED_CHURNER=%.2f, COST_WASTED_OFFER=%.2f, ratio=%.2f:1",
            COST_MISSED_CHURNER, COST_WASTED_OFFER, COST_MISSED_CHURNER / COST_WASTED_OFFER)
logger.info("MIN_COST_REDUCTION_VS_TRIVIAL=%.0f%%", 100 * MIN_COST_REDUCTION_VS_TRIVIAL)

00:15:56 | 06-comparison-threshold | INFO | COST_MISSED_CHURNER=1.00, COST_WASTED_OFFER=0.15, ratio=6.67:1


00:15:56 | 06-comparison-threshold | INFO | MIN_COST_REDUCTION_VS_TRIVIAL=20%


## 1. Load everything

No retraining here. Loading the train/test split and the three fitted models from
Stages 6 and 7. XGBoost and LightGBM already have saved out-of-fold predictions;
Logistic Regression doesn't, since Stage 7 only needed its score, not its predictions.

Expecting all three models to load and the two saved files to have 4,204 rows each, one
per training customer.

In [3]:
train_df = pd.read_parquet(TRAIN_PARQUET)
test_df = pd.read_parquet(TEST_PARQUET)
X_train, y_train = train_df[list(FEATURE_COLUMNS)], train_df["y"]
X_test, y_test = test_df[list(FEATURE_COLUMNS)], test_df["y"]

logreg_pipeline = joblib.load(MODELS_DIR / "baseline_logreg.joblib")
xgb_pipeline = joblib.load(MODELS_DIR / "xgboost.joblib")
lgbm_pipeline = joblib.load(MODELS_DIR / "lightgbm.joblib")

xgb_oof = pd.read_parquet(OUTPUT_DIR / "xgboost_oof_probabilities.parquet")
lgbm_oof = pd.read_parquet(OUTPUT_DIR / "lightgbm_oof_probabilities.parquet")

logger.info("train/test: %d / %d rows", len(train_df), len(test_df))
logger.info("xgboost OOF rows: %d", len(xgb_oof))
logger.info("lightgbm OOF rows: %d", len(lgbm_oof))
assert len(xgb_oof) == len(train_df) and len(lgbm_oof) == len(train_df)
assert (xgb_oof["customer_id"].values == train_df["customer_id"].values).all()
assert (lgbm_oof["customer_id"].values == train_df["customer_id"].values).all()

00:15:57 | 06-comparison-threshold | INFO | train/test: 4204 / 1052 rows


00:15:57 | 06-comparison-threshold | INFO | xgboost OOF rows: 4204


00:15:57 | 06-comparison-threshold | INFO | lightgbm OOF rows: 4204


Everything loaded and lined up correctly. The cost ratio is 6.67:1: missing a churner
costs about 6.7 times what a wasted discount costs. That imbalance is why the best
threshold will land well below 0.5, not at it.

## 2. Fair scores for every candidate, RFM included

Two candidates are missing a fair, leakage-free score:

- **Logistic Regression** only has a CV *score* from Stage 7, not per-customer
  predictions. Generating those now with the same 5-fold split.
- **RFM was never scored fairly at all.** Stage 6 fit its rule on the training set and
  then graded it on that same set. This time: fit the rule on 4/5 of the data, grade it
  on the held-out 1/5, same as the models get graded. This is the decision layer for the
  whole project, so RFM gets held to the same standard as everything else.

Expecting Logistic Regression to land close to Stage 7's 0.7946, and RFM close to its
old 0.7932, since a simple 3-feature rule shouldn't move much either way.

In [4]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

logreg_oof_proba = cross_val_predict(logreg_pipeline, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
logger.info("Logistic Regression OOF ROC-AUC: %.4f (Stage 7 CV mean: 0.7946)",
            roc_auc_score(y_train, logreg_oof_proba))


def fit_rfm_bins(train_series: pd.Series, ascending: bool):
    _, edges = pd.qcut(train_series, 5, retbins=True, duplicates="drop")
    edges = edges.copy()
    edges[0], edges[-1] = -np.inf, np.inf
    n_bins = len(edges) - 1
    labels = list(range(1, n_bins + 1)) if ascending else list(range(n_bins, 0, -1))
    return edges, labels


def apply_rfm_bins(series: pd.Series, edges, labels) -> pd.Series:
    return pd.cut(series, bins=edges, labels=labels, include_lowest=True).astype(int)


rfm_cols = {"recency_days": False, "frequency": True, "monetary_total": True}
rfm_oof_score = pd.Series(index=X_train.index, dtype=float)

for fold_train_idx, fold_val_idx in cv.split(X_train, y_train):
    fold_train = X_train.iloc[fold_train_idx]
    fold_val = X_train.iloc[fold_val_idx]
    fold_bins = {col: fit_rfm_bins(fold_train[col], asc) for col, asc in rfm_cols.items()}
    fold_score = sum(apply_rfm_bins(fold_val[col], *fold_bins[col]) for col in rfm_cols)
    rfm_oof_score.iloc[fold_val_idx] = fold_score.values

logger.info("RFM OOF ROC-AUC: %.4f (Stage 6 single-split: 0.7932)", roc_auc_score(y_train, rfm_oof_score))

00:15:57 | 06-comparison-threshold | INFO | Logistic Regression OOF ROC-AUC: 0.7942 (Stage 7 CV mean: 0.7946)


00:15:57 | 06-comparison-threshold | INFO | RFM OOF ROC-AUC: 0.7811 (Stage 6 single-split: 0.7932)


Logistic Regression matches as expected (0.7942 vs 0.7946). RFM doesn't: **0.7811, down
from 0.7932.** That's a bigger drop than "shouldn't move much" predicted, so worth
checking why instead of just accepting it.

In [5]:
for fold_i, (fold_train_idx, fold_val_idx) in enumerate(cv.split(X_train, y_train)):
    fold_train = X_train.iloc[fold_train_idx]
    fold_val = X_train.iloc[fold_val_idx]
    fold_bins = {col: fit_rfm_bins(fold_train[col], asc) for col, asc in rfm_cols.items()}
    n_groups = {col: len(labels) for col, (edges, labels) in fold_bins.items()}
    fold_score = sum(apply_rfm_bins(fold_val[col], *fold_bins[col]) for col in rfm_cols)
    fold_auc = roc_auc_score(y_train.iloc[fold_val_idx], fold_score)
    logger.info("fold %d: groups=%s, fold ROC-AUC=%.4f", fold_i, n_groups, fold_auc)

logger.info("in-sample (Stage 6 style, bins fit and scored on the same full train set): %.4f",
            roc_auc_score(y_train, sum(apply_rfm_bins(X_train[col], *fit_rfm_bins(X_train[col], asc))
                                        for col, asc in rfm_cols.items())))

00:15:57 | 06-comparison-threshold | INFO | fold 0: groups={'recency_days': 5, 'frequency': 4, 'monetary_total': 5}, fold ROC-AUC=0.7776


00:15:57 | 06-comparison-threshold | INFO | fold 1: groups={'recency_days': 5, 'frequency': 4, 'monetary_total': 5}, fold ROC-AUC=0.7940


00:15:57 | 06-comparison-threshold | INFO | fold 2: groups={'recency_days': 5, 'frequency': 4, 'monetary_total': 5}, fold ROC-AUC=0.7951


00:15:57 | 06-comparison-threshold | INFO | fold 3: groups={'recency_days': 5, 'frequency': 4, 'monetary_total': 5}, fold ROC-AUC=0.7640


00:15:57 | 06-comparison-threshold | INFO | fold 4: groups={'recency_days': 5, 'frequency': 4, 'monetary_total': 5}, fold ROC-AUC=0.7759


00:15:57 | 06-comparison-threshold | INFO | in-sample (Stage 6 style, bins fit and scored on the same full train set): 0.7812


**Not leakage.** Scoring RFM on the exact same rows its rule was fit on (what Stage 6
effectively did, but here computed on train instead of test) gives 0.7812, basically
identical to the proper fair score of 0.7811. If fitting-on-the-scored-rows were
inflating anything, this in-sample number would sit well above the fair one. It doesn't.

**The real explanation: the old test split was just lucky for RFM too**, the same thing
Stage 7 already found for the three models (they all scored higher on that test split
than in cross-validation). RFM's honest performance is 0.781; 0.7932 was a good day for
one specific 1,052-customer split. Four different candidates now agree that split runs
a little easy across the board, so every comparison from here uses the fair score, not
the old test-split number.

### 2a. Redoing the "beats RFM" check with the honest number

Stage 7 compared each model against RFM's lucky 0.7932 and said nothing cleared the bar.
That wasn't a fair fight: the models were held to a fair standard, RFM wasn't. Redoing it
with RFM's real 0.7811 on both sides.

In [6]:
rfm_oof_auc = roc_auc_score(y_train, rfm_oof_score)
logreg_oof_auc = roc_auc_score(y_train, logreg_oof_proba)
xgb_oof_auc = roc_auc_score(y_train, xgb_oof["oof_proba"])
lgbm_oof_auc = roc_auc_score(y_train, lgbm_oof["oof_proba"])

# Stage 7's fold standard deviations, unaffected by this correction (they describe each
# ML model's own variability, not RFM's).
stds = {"Logistic Regression": 0.0114, "LightGBM": 0.0093, "XGBoost": 0.0107}
oof_aucs = {"Logistic Regression": logreg_oof_auc, "LightGBM": lgbm_oof_auc, "XGBoost": xgb_oof_auc}

logger.info("RFM honest OOF ROC-AUC: %.4f", rfm_oof_auc)
for name in stds:
    gap = oof_aucs[name] - rfm_oof_auc
    logger.info("%-20s OOF %.4f | gap over RFM %+.4f | own std %.4f | beats RFM: %s",
                name, oof_aucs[name], gap, stds[name], gap > stds[name])

00:15:57 | 06-comparison-threshold | INFO | RFM honest OOF ROC-AUC: 0.7811


00:15:57 | 06-comparison-threshold | INFO | Logistic Regression  OOF 0.7942 | gap over RFM +0.0131 | own std 0.0114 | beats RFM: True


00:15:57 | 06-comparison-threshold | INFO | LightGBM             OOF 0.7979 | gap over RFM +0.0168 | own std 0.0093 | beats RFM: True


00:15:57 | 06-comparison-threshold | INFO | XGBoost              OOF 0.8018 | gap over RFM +0.0207 | own std 0.0107 | beats RFM: True


**Correction: all three models actually do beat RFM now.**

| Model | Score | Gap over RFM (0.7811) | Beats RFM |
|---|---|---|---|
| Logistic Regression | 0.7942 | +0.0131 | **Yes** |
| LightGBM | 0.7979 | +0.0168 | **Yes** |
| XGBoost | 0.8018 | +0.0207 | **Yes** |

Stage 7 wasn't wrong about the math, just about what it was comparing against. Once RFM
gets graded the same fair way, every model beats it, and XGBoost's edge (+0.0207) is
roughly double the amount of noise it would need to clear. **XGBoost is still the best
of the three**, same as Stage 7 found.

## 3. Threshold tuning by cost, not accuracy

ROC-AUC just picked the model. It says nothing about *where to draw the line* between
"send an offer" and "leave alone", and 0.5 is not a business decision.

Rule: send an offer when a customer's score is below the threshold. Two ways to be
wrong, each with its own price:

- **Missed churner** (costs 1.0): score says "will repeat" but they don't. No offer sent.
- **Wasted offer** (costs 0.15): score says "won't repeat" but they would have. Offer
  sent for nothing.

Testing every possible threshold using train scores only, never test. Works the same way
for the three probability scores (0 to 1) and RFM's score (3 to 14): higher always means
more likely to repeat.

In [7]:
def expected_cost(y_true: np.ndarray, score: np.ndarray, threshold: float) -> float:
    """Total expected cost of sending offers to everyone scoring below `threshold`."""
    send_offer = score < threshold
    missed_churners = ((~send_offer) & (y_true == 0)).sum()
    wasted_offers = (send_offer & (y_true == 1)).sum()
    return COST_MISSED_CHURNER * missed_churners + COST_WASTED_OFFER * wasted_offers


def sweep_threshold(y_true: np.ndarray, score: np.ndarray) -> tuple[float, float]:
    """Return (best_threshold, min_cost) over every candidate cut point in `score`."""
    candidates = np.unique(score)
    costs = [expected_cost(y_true, score, t) for t in candidates]
    best_idx = int(np.argmin(costs))
    return float(candidates[best_idx]), float(costs[best_idx])


candidates_oof = {
    "RFM heuristic": rfm_oof_score.values,
    "Logistic Regression": logreg_oof_proba,
    "LightGBM": lgbm_oof["oof_proba"].values,
    "XGBoost": xgb_oof["oof_proba"].values,
}

threshold_results = {}
for name, score in candidates_oof.items():
    best_t, min_cost = sweep_threshold(y_train.values, score)
    threshold_results[name] = {"threshold": best_t, "oof_cost": min_cost}
    logger.info("%-20s best threshold=%.4f | min OOF cost=%.2f | cost per customer=%.4f",
                name, best_t, min_cost, min_cost / len(y_train))

00:15:57 | 06-comparison-threshold | INFO | RFM heuristic        best threshold=14.0000 | min OOF cost=254.75 | cost per customer=0.0606


00:15:57 | 06-comparison-threshold | INFO | Logistic Regression  best threshold=0.8500 | min OOF cost=245.70 | cost per customer=0.0584


00:15:57 | 06-comparison-threshold | INFO | LightGBM             best threshold=0.9106 | min OOF cost=242.55 | cost per customer=0.0577


00:15:57 | 06-comparison-threshold | INFO | XGBoost              best threshold=0.8878 | min OOF cost=240.05 | cost per customer=0.0571


**XGBoost is cheapest (0.0571 per customer). Same order as everywhere else: XGBoost <
LightGBM < Logistic Regression < RFM.**

Every threshold lands high. RFM's best cutoff is basically "offer everyone except the
very top score." The three models cluster around 0.85 to 0.91. That's the cost ratio
working as intended: since a wasted offer is cheap and a missed churner is expensive,
"when in doubt, send the offer" wins for all four candidates.

One caveat: RFM only has 12 possible score values to choose a threshold from, versus
roughly 4,200 for the models. Landing at the extreme might partly be "no finer option
available," not only "worse ranking." Either way, RFM's best possible outcome still
costs more than any model's.

## 4. Apply to test, exactly once

Every threshold so far came from train data. Test gets touched now, once, to confirm
the decision already made: XGBoost, threshold 0.8878.

The real bar (from `docs/problem_definition.md`): **a 20 percent cost reduction against
the best of three fallback options**: contact nobody, contact everybody, or RFM. This
is the actual pass/fail test for the whole project, not ROC-AUC.

In [8]:
SELECTED_MODEL = "XGBoost"
selected_threshold = threshold_results[SELECTED_MODEL]["threshold"]

xgb_test_proba = xgb_pipeline.predict_proba(X_test)[:, 1]
selected_test_cost = expected_cost(y_test.values, xgb_test_proba, selected_threshold)

# Trivial policies, evaluated on test.
cost_contact_nobody = COST_MISSED_CHURNER * (y_test == 0).sum()
cost_contact_everybody = COST_WASTED_OFFER * (y_test == 1).sum()

train_rfm_fit = {col: fit_rfm_bins(X_train[col], asc) for col, asc in rfm_cols.items()}
test_rfm_score = sum(apply_rfm_bins(X_test[col], *train_rfm_fit[col]) for col in rfm_cols)
cost_rfm_test = expected_cost(y_test.values, test_rfm_score.values, threshold_results["RFM heuristic"]["threshold"])

trivial_costs = {
    "Contact nobody": cost_contact_nobody,
    "Contact everybody": cost_contact_everybody,
    "RFM heuristic (its own best threshold)": cost_rfm_test,
}
best_trivial_name = min(trivial_costs, key=trivial_costs.get)
best_trivial_cost = trivial_costs[best_trivial_name]

cost_reduction = 1 - (selected_test_cost / best_trivial_cost)

logger.info("XGBoost test cost at threshold %.4f: %.2f (%.4f per customer)",
            selected_threshold, selected_test_cost, selected_test_cost / len(y_test))
for name, cost in trivial_costs.items():
    logger.info("  %-40s cost=%.2f (%.4f per customer)", name, cost, cost / len(y_test))
logger.info("best trivial policy: %s (%.2f)", best_trivial_name, best_trivial_cost)
logger.info("cost reduction vs best trivial policy: %.1f%% (bar: %.0f%%)",
            100 * cost_reduction, 100 * MIN_COST_REDUCTION_VS_TRIVIAL)
logger.info("clears the Stage 0 bar: %s", cost_reduction >= MIN_COST_REDUCTION_VS_TRIVIAL)

00:15:57 | 06-comparison-threshold | INFO | XGBoost test cost at threshold 0.8878: 61.95 (0.0589 per customer)


00:15:57 | 06-comparison-threshold | INFO |   Contact nobody                           cost=594.00 (0.5646 per customer)


00:15:57 | 06-comparison-threshold | INFO |   Contact everybody                        cost=68.70 (0.0653 per customer)


00:15:57 | 06-comparison-threshold | INFO |   RFM heuristic (its own best threshold)   cost=64.45 (0.0613 per customer)


00:15:57 | 06-comparison-threshold | INFO | best trivial policy: RFM heuristic (its own best threshold) (64.45)


00:15:57 | 06-comparison-threshold | INFO | cost reduction vs best trivial policy: 3.9% (bar: 20%)


00:15:57 | 06-comparison-threshold | INFO | clears the Stage 0 bar: False


**XGBoost doesn't clear the bar. 3.9 percent cost reduction, not the required 20.**

| Policy | Cost per customer |
|---|---|
| Contact nobody | 0.5646 |
| Contact everybody | 0.0653 |
| RFM (best threshold) | **0.0613** |
| XGBoost (best threshold) | **0.0589** |

RFM beats both "do nothing" options and is the real bar to clear here, not the other
two. XGBoost only edges past it by a small margin.

Both of these are true at once: **XGBoost really does rank customers better than RFM**
(section 2a, a real gap, not noise), **but that better ranking barely shows up as
savings** at this cost ratio. Every candidate's best move is close to "offer almost
everyone" (section 3). Once nearly everyone is already getting an offer, sharper ranking
has very little room left to save money.

### 4a. What this actually looks like for the retention team

How many offers go out, how many churners get caught, how many get missed. Expecting a
lot of offers, since the threshold sits near the top of the score range.

In [9]:
send_offer_test = xgb_test_proba < selected_threshold
n_offers = send_offer_test.sum()
n_churners_total = (y_test == 0).sum()
n_churners_caught = (send_offer_test & (y_test == 0)).sum()
n_churners_missed = ((~send_offer_test) & (y_test == 0)).sum()
n_wasted = (send_offer_test & (y_test == 1)).sum()

logger.info("offers sent: %d of %d customers (%.1f%%)", n_offers, len(y_test), 100 * n_offers / len(y_test))
logger.info("churners caught: %d of %d (%.1f%% of all churners)",
            n_churners_caught, n_churners_total, 100 * n_churners_caught / n_churners_total)
logger.info("churners missed: %d", n_churners_missed)
logger.info("wasted offers (would have repeated anyway): %d", n_wasted)
logger.info("precision of the offer list (share who were actually churners): %.1f%%",
            100 * n_churners_caught / n_offers)

00:15:57 | 06-comparison-threshold | INFO | offers sent: 984 of 1052 customers (93.5%)


00:15:57 | 06-comparison-threshold | INFO | churners caught: 591 of 594 (99.5% of all churners)


00:15:57 | 06-comparison-threshold | INFO | churners missed: 3


00:15:57 | 06-comparison-threshold | INFO | wasted offers (would have repeated anyway): 393


00:15:57 | 06-comparison-threshold | INFO | precision of the offer list (share who were actually churners): 60.1%


**93.5 percent of customers get an offer. 591 of 594 actual churners are caught, only 3
missed. But 393 of the 984 offers (40 percent) go to people who'd have come back
anyway.**

That's the cost ratio at work, not a flaw. Missing a churner costs 6.7 times a wasted
offer, so the safest policy offers almost everyone something. The model's real
contribution: it confidently rules out about 6.5 percent of customers (68 people) as
"definitely coming back," saving that spend without missing more churners.

A real but small win. Same story as the 3.9 percent cost reduction above: not nothing,
not enough to justify replacing the simpler rule.

## 5. Save the final record

`models/model_selection.json` records XGBoost as the best candidate found, plus the
honest conclusion in plain words: the cost bar wasn't met, so there's no case for
deploying it.

In [10]:
import json
from datetime import UTC, datetime

meets_bar = cost_reduction >= MIN_COST_REDUCTION_VS_TRIVIAL

selection_record = {
    "selected_model": SELECTED_MODEL,
    "model_path": str((MODELS_DIR / "xgboost.joblib").relative_to(PROJECT_ROOT)),
    "threshold": selected_threshold,
    "decision_parameters": {
        "cost_missed_churner": COST_MISSED_CHURNER,
        "cost_wasted_offer": COST_WASTED_OFFER,
        "min_cost_reduction_required": MIN_COST_REDUCTION_VS_TRIVIAL,
    },
    "metrics": {
        "oof_roc_auc": {
            "rfm_heuristic": rfm_oof_auc,
            "logistic_regression": logreg_oof_auc,
            "lightgbm": lgbm_oof_auc,
            "xgboost": xgb_oof_auc,
        },
        "test_cost_per_customer": {
            "contact_nobody": float(cost_contact_nobody / len(y_test)),
            "contact_everybody": float(cost_contact_everybody / len(y_test)),
            "rfm_heuristic": float(cost_rfm_test / len(y_test)),
            "xgboost_selected": float(selected_test_cost / len(y_test)),
        },
        "cost_reduction_vs_best_trivial_policy": cost_reduction,
        "best_trivial_policy": best_trivial_name,
        "test_offers_sent_pct": float(100 * n_offers / len(y_test)),
        "test_churners_caught_pct": float(100 * n_churners_caught / n_churners_total),
        "test_offer_precision_pct": float(100 * n_churners_caught / n_offers),
    },
    "meets_cost_reduction_bar": bool(meets_bar),
    "recommendation": (
        f"Deploy XGBoost at threshold {selected_threshold:.4f}." if meets_bar else
        "Do not deploy a model. XGBoost is the best candidate found and its ranking "
        "advantage over the RFM heuristic is real (exceeds cross-validation noise), but "
        "the resulting cost reduction (3.9 percent) falls short of the 20 percent bar "
        "this project set for justifying the added complexity of a trained model over "
        "the existing RFM rule. Recommend the retention team continue using the RFM "
        "heuristic, or adopt XGBoost only if the added complexity is independently "
        "judged worthwhile despite the small measured benefit."
    ),
    "generated_at": datetime.now(UTC).isoformat(),
}

with open(MODEL_SELECTION_JSON, "w") as f:
    json.dump(selection_record, f, indent=2)

logger.info("wrote %s", MODEL_SELECTION_JSON.name)
logger.info("meets_cost_reduction_bar: %s", meets_bar)

00:15:57 | 06-comparison-threshold | INFO | wrote model_selection.json


00:15:57 | 06-comparison-threshold | INFO | meets_cost_reduction_bar: False


Saved. `meets_cost_reduction_bar: false`, with the reasoning spelled out in plain
sentences, not just numbers, so it's still clear on its own months from now.

## 6. Stage 8 summary, and project conclusion

### The decision

**Don't deploy a model. Keep using RFM.** XGBoost ranks customers better than RFM in a
way that's real, not noise, but the money it saves (3.9 percent) falls well short of
the 20 percent bar this project set before deployment made sense. Full writeup in
`docs/model_selection.md`.

### What got saved

- `models/model_selection.json`: the full record, with a plain-language recommendation.
- `docs/model_selection.md`: the decision writeup.
- Fair, leakage-free scores for all four candidates, RFM included.

### The two corrections this stage made

1. **Stage 7 compared RFM unfairly.** It graded the three models honestly but graded RFM
   on a lucky test split. Once RFM gets the same fair treatment, its score drops from
   0.7932 to 0.7811, and all three models turn out to beat it after all.
2. **That still wasn't enough.** Beating RFM on ranking (ROC-AUC) didn't translate into
   beating it on the metric that actually matters: money saved. The project's own
   three-part bar from Stage 0 is what caught this.

### Key numbers

| Candidate | Fair score (ROC-AUC) | Test cost per customer |
|---|---|---|
| RFM heuristic | 0.7811 | 0.0613 |
| Logistic Regression | 0.7942 | not selected |
| LightGBM | 0.7979 | not selected |
| XGBoost | **0.8018** | **0.0589** |

### Exit check

| Requirement | Status |
|---|---|
| Models loaded, nothing retrained | Section 1 |
| Threshold tuned on train, applied to test once | Sections 3-4 |
| Model picked on fair train scores, not the test table | Section 2a |
| Real-world error breakdown at the chosen threshold | Section 4a |
| Selection file saved | Section 5 |
| Decision doc written | `docs/model_selection.md` |

Project ends here, at Stage 8, by choice. Explainability, final export, and serving
(Stages 9 to 11) are out of scope, see `CLAUDE.md`.